In [16]:
import torch
import re
import random
from torch import nn
import math
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter

In [17]:
##参数初始化
def get_params(vocab_size, num_hiddens, device):

    #vocabsize表示一个输入x向量的长度，
    #numhiddens表示一个隐藏层向量w的长度
    num_inputs = num_outputs = vocab_size
    #输入是one-hot向量，输出也是


    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01

    # 隐藏层参数
    ##将输入映射到隐藏空间
    W_xh = normal((num_inputs, num_hiddens))
    
    ##用于处理上一个时刻的隐藏状态H
    W_hh = normal((num_hiddens, num_hiddens))

    ##隐藏层偏置
    b_h = torch.zeros(num_hiddens, device=device)

    ##将隐藏状态转换为词表每个字符预测的分数
    W_hq = normal((num_hiddens, num_outputs))
    ##输出层偏置
    b_q = torch.zeros(num_outputs, device=device)
    # 附加梯度
    params = [W_xh, W_hh, b_h, W_hq, b_q]

    ##显示表示需要梯度
    for param in params:
        param.requires_grad_(True)
    return params

In [18]:
##初始化隐变量
def init_rnn_state(batch_size, num_hiddens, device):
    #每个样本都会形成自己专有的隐变量状态
    return (torch.zeros((batch_size, num_hiddens), device=device), )

In [19]:
#RNN计算步骤
def rnn(inputs, state, params):
    #时间步指到第几个隐状态
    # inputs的形状：(时间步数量，批量大小，词表大小)
    W_xh, W_hh, b_h, W_hq, b_q = params

    ##state的第0维是[batch_size,num_hidden]
    ##也就是矩阵H的维度
    ##传入初始状态
    H, = state
    outputs = []

    ##每次取出一个时间步的数据
    # X的形状：(批量大小，词表大小)
    ##torch.mm(X, W_xh)计算完的形状为[batch_size,num_hiddens]
    for X in inputs:
        H = torch.tanh(torch.mm(X, W_xh) + torch.mm(H, W_hh) + b_h)
        Y = torch.mm(H, W_hq) + b_q
        outputs.append(Y)
    ##这里W的形状是[input_size,num_hiddens]
    ##这里输出的Y的形状为[batchsize,vocabsize]
    ##output的形状为[时间步，batchsize，vocabsize]
    ##不同时间步的batchsize融合到一块
    return torch.cat(outputs, dim=0), (H,)
    ##将所有时间步合到一块，便于计算损失函数
    ##传回H，作为下一个batch的初始状态
    ##这里传回（H，）是为了统一接口，因为LSTM有两个潜变量

In [20]:
#封装RNN，进行one-hot编码
class RNNModelScratch: #@save
    """从零开始实现的循环神经网络模型"""
    def __init__(self, vocab_size, num_hiddens, device,
                 get_params, init_state, forward_fn):
        self.vocab_size, self.num_hiddens = vocab_size, num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state, self.forward_fn = init_state, forward_fn

    def __call__(self, X, state):
        ##先将X转置，[bactusize,时间步]转为[时间步,batchsize]
        ##再进行onehot编码，X中的值只能在vocabsize内
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)

    ##初始化隐变量
    def begin_state(self, batch_size, device):
        return self.init_state(batch_size, self.num_hiddens, device)

In [21]:
#vocab的实现，用于将字母转为数字，将数字转为字母
class Vocab:
    def __init__(self, tokens):
        """
        参数:
            tokens: 所有字符的列表，例如 ['a', 'b', 'c', ...]
        """
        # 去重并排序
        unique_tokens = sorted(set(tokens))
        
        # 建立索引映射
        self.idx_to_token = unique_tokens
        self.token_to_idx = {token: idx for idx, 
                             token in enumerate(unique_tokens)}
    
    def __getitem__(self, token):
        """支持 vocab[token] 语法，返回 token 对应的索引"""
        return self.token_to_idx[token]


    def __len__(self):
        """返回词表大小"""
        return len(self.idx_to_token)

In [22]:
#定义预测函数
def predict_ch8(prefix, 
                num_preds, net, vocab, device):
    #prefix是给定的开头字符串
    #num_pred是要接着生成的字符
    #vocab负责字符和符号之间的转换（one-hot编码）

    """在prefix后面生成新字符"""
    #初始化隐藏状态
    ##在推测时，只关注一个句子，所以batchsize=1即可
    state = net.begin_state(batch_size=1, device=device)

    ##将prefix的第一个字符转换成词表索引，并放进output
    outputs = [vocab[prefix[0]]]

    #把output里边最后一个元素取出，形状改为（1，1）
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))

    ##预热结束后output中是prefix的所有字符索引
    for y in prefix[1:]:  # 预热期
        _, state = net(get_input(), state)
        outputs.append(vocab[y])

    ##从prefix最后一个字符索引开始预测
    for _ in range(num_preds):  # 预测num_preds步
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return ''.join([vocab.idx_to_token[i] for i in outputs])

In [23]:
#定义梯度裁剪
def grad_clipping(net, theta):  #@save
    """裁剪梯度"""
    ##获取参数param
    ##如果是继承了nn.Moudule
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params

    ##如果参数的和超过了θ，所有的参数都×
    ##θ/norm
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

In [24]:
#读取并清洗数据集
def read_time_machine(file_path='timemachine.txt'):
    """读取时间机器数据集，并做基本文本清洗"""

    ##打开文件，取每一行
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # 只保留英文字母，将连续空白合并为一个空格，并转换为小写
    lines = [
        re.sub(r'[^A-Za-z]+', ' ', line).strip().lower()
        for line in lines
    ]

    # 去掉清洗后的空行
    return [line for line in lines if line]
#加载数据集，返回词表对象和数字索引列表
def load_corpus_time_machine(file_path='timemachine.txt'):
    """
    返回：
        corpus: 文本中每个字符对应的词表索引
        vocab:  字符词表
    """
    #max_token用于限制数据量，加快训练速度
    lines = read_time_machine(file_path)

    # 将所有行连接起来，并保留单词之间的空格
    text = ' '.join(lines)

    # 字符级建模
    tokens = list(text)

    vocab = Vocab(tokens)
    corpus = [vocab[token] for token in tokens]
    return corpus, vocab

In [25]:
#获得输入X和输出Y，Y在corpus上落后X一位
##X和Y还没有进行onehoot编码
def seq_data_iter_random(corpus, batch_size, num_steps):
    """
    使用随机采样生成小批量序列。

    X 和 Y 的形状均为：
        [batch_size, num_steps]

    Y 是 X 向后移动一个字符得到的标签。
    """
    # 随机选择起点
    ##使得每次切分的起点不同，增加样本多样性
    ##从0到numstep-1
    offset = random.randint(0, num_steps - 1)
    corpus = corpus[offset:]

    ##每个随机取的corpus可以构成多少个满足时间步的序列
    num_subseqs = (len(corpus) - 1) // num_steps

    # 每个子序列的起始位置
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    ##对起始位置序列进行打乱
    random.shuffle(initial_indices)

    def data(pos):
        return corpus[pos:pos + num_steps]

    ##获得的时间步序列能有几个batchsize
    num_batches = num_subseqs // batch_size

    for i in range(0, batch_size * num_batches, batch_size):
        ##在起始序列中拿batchsize个
        batch_indices = initial_indices[i:i + batch_size]

        ##获得X和Y的序列
        X = [data(j) for j in batch_indices]
        Y = [data(j + 1) for j in batch_indices]

        ##yield不断返回batchsizeX和Y的tensor的向量
        ##一次返回一个batchsize，直到下回调用
        yield (
            torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long)
        )

In [26]:
##封装数据加载器
class SeqDataLoader:
    """时间机器数据集的字符级序列批量加载器"""

    def __init__(self,
                 batch_size,
                 num_steps,
                 file_path='timemachine.txt'):
        self.corpus, self.vocab = load_corpus_time_machine(
            file_path=file_path
        )

        self.batch_size = batch_size
        self.num_steps = num_steps

    def __iter__(self):
        return seq_data_iter_random(
            self.corpus,
            self.batch_size,
            self.num_steps
        )
##创建迭代器，返回迭代器（和vocab
def load_data_time_machine(batch_size,
                           num_steps,
                           file_path='timemachine.txt'):
    """创建数据迭代器和词表"""
    data_iter = SeqDataLoader(
        batch_size=batch_size,
        num_steps=num_steps,
        file_path=file_path,
    )

    return data_iter, data_iter.vocab

In [ ]:
#单个循环轮次的训练
def train_epoch_rnn(net,
                    data_iter,
                    loss_fn,
                    optimizer,
                    device,
                    clip_theta=1.0):
    """
    训练一个 epoch。

    返回：
        平均损失
        困惑度 perplexity
    """
    total_loss = 0.0
    total_tokens = 0

    # 随机采样时，每个批次互不连续，
    # 因此每个批次都重新初始化隐藏状态
    for X, Y in data_iter:
        X = X.to(device)
        Y = Y.to(device)

        batch_size = X.shape[0]


        ##每次重新初始化隐状态
        state = net.begin_state(
            batch_size=batch_size,
            device=device
        )

        # 清除上一个批次留下的梯度
        optimizer.zero_grad()

        # 前向传播
        y_hat, state = net(X, state)

        # y_hat 的顺序是：
        # 时间步0的整个batch、时间步1的整个batch……
        # 因此标签也必须先转置为 [时间步, batch_size]
        ##然后展开[时间步*bathcsize]
        y = Y.T.reshape(-1)

        # 计算交叉熵损失
        ##计算交叉熵时y是2，表示真实为2
        ##yhat[0,0,1,0],只会计算2的值
        loss = loss_fn(y_hat, y)

        # 反向传播
        loss.backward()

        # 防止 RNN 梯度爆炸
        grad_clipping(net, clip_theta)

        # SGD 参数更新
        optimizer.step()

        ##函数numel是获得目标的所有张量
        ##    在这里是获得预测的batchsize*numstep
        num_tokens = y.numel()

        #一轮的总损失
        total_loss += loss.item() * num_tokens
        #总共预测的数量
        total_tokens += num_tokens

    average_loss = total_loss / total_tokens
    perplexity = math.exp(min(average_loss, 20))

    return average_loss, perplexity
##封装整体训练
def train_rnn(net,
              data_iter,
              vocab,
              device,
              num_epochs,
              log_dir,
              learning_rate=0.01,
              clip_theta=1.0,
              predict_prefix='time traveller',
              num_preds=20):
    """训练字符级 RNN"""
    loss_fn = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(
        net.params,
        lr=learning_rate
    )
    writer = SummaryWriter(log_dir=log_dir)
    for epoch in range(1, num_epochs + 1):
        average_loss, perplexity = train_epoch_rnn(
            net=net,
            data_iter=data_iter,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device,
            clip_theta=clip_theta
        )
        print(f'=========epoch{epoch}===========')

        writer.add_scalar(
                'Train/Loss',
                average_loss,
                epoch
            )

        writer.add_scalar(
                'Train/Perplexity',
                perplexity,
                epoch
            )
        generated_text = predict_ch8(prefix=predict_prefix,
                                     num_preds=num_preds,
                                     net = net,
                                     vocab = vocab,
                                     device=device)
        print(f'预测文本：{generated_text}')
    writer.close()


In [28]:
##加载数据,获取迭代器和字母数字映射
batch_size = 32
num_steps = 35

data_iter, vocab = load_data_time_machine(
    batch_size=batch_size,
    num_steps=num_steps,
    file_path='timemachine.txt',
)

##选择设备
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

num_hiddens = 512

##定义RNN网络
net = RNNModelScratch(
    vocab_size=len(vocab),
    num_hiddens=num_hiddens,
    device=device,
    get_params=get_params,
    init_state=init_rnn_state,
    forward_fn=rnn
)

##开始训练
num_epochs = 1000
logdir = './logs/RNN_first'
train_rnn(
    net=net,
    data_iter=data_iter,
    vocab=vocab,
    device=device,
    num_epochs=num_epochs,
    clip_theta=1.0,
    predict_prefix='time traveller',
    num_preds=20,log_dir=logdir
)

=========epoch1===========
预测文本：time traveller                                                  
=========epoch2===========
预测文本：time traveller                                                  
=========epoch3===========
预测文本：time traveller                                                  
=========epoch4===========
预测文本：time traveller                                                  
=========epoch5===========
预测文本：time traveller                                                  
=========epoch6===========
预测文本：time traveller                                                  
=========epoch7===========
预测文本：time traveller                                                  
=========epoch8===========
预测文本：time traveller                                                  
=========epoch9===========
预测文本：time traveller                                                  
=========epoch10===========
预测文本：time traveller                                                  
=========epoch11===========
预